# Datenqualität: unplausible Aufnahmen und Labels

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Die Kickoff-Herausforderung „irreguläre Aufnahmen“ prüfen (#196). Verdächtige Zeilen werden **markiert, nicht entfernt**. Dann klären wir, ob sich die Fehlklassifikationen dort häufen. Nur dann lohnt das Entfernen im Training (#197).

**Wichtig zur Skala:** Blau liegt im Median bei rund 16 % Reflektanz, deutlich über den 3–6 % von Vegetation an der Oberfläche. Absolute Schwellen aus der Literatur (z. B. der Dunst-Test von Fmask) passen deshalb nicht. Alle spektralen Regeln markieren nur Tukeys „far out“-Werte: unter Q1 − 3·IQR oder über Q3 + 3·IQR.

| Markierung | Regel | Findet |
| --- | --- | --- |
| `haze` | HOT-Wert (Blau − 0,5·Rot) extrem hoch, über alle Zeilen | Dunst, dünne Wolken (Zhu & Woodcock 2012) |
| `dark` | NIR (860 nm) extrem niedrig, über alle Zeilen | Schatten, offenes Wasser |
| `ndvi_outlier` | NDVI extrem für die eigene Crop/Stage-Klasse | z. B. „Late“ mit Boden-NDVI |
| `rare_context` | Monat × Kultur × Stadium höchstens 3× im Training | phänologisch unplausible Labels |
| `class_outlier` | robuste Mahalanobis-Distanz (5 PCA-Komponenten) extrem für die eigene Klasse | Spektren weit weg von ihrer Klasse |
| Label-Verdacht | SVM und Extra Trees sagen out-of-fold übereinstimmend dieselbe andere Klasse vorher | falsche Labels (Konsens-Filter, Brodley & Friedl 1999) |

Ausgewertet wird nur der Trainingsteil (80/20-Split); der Validierungsteil bleibt unberührt.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import (
    AEZ_COL,
    ANY_FLAG_COL,
    LABEL_SUSPECT_COL,
    MODEL_DOCS_FIGURES_DIR,
    MONTH_COL,
    SPECTRAL_REGIONS,
)
from awp2.data import band_columns, combined_label, load_split, wavelengths
from awp2.experiment import RunConfig, out_of_fold_predictions
from awp2.models import make_combined_extra_trees, make_combined_svm
from awp2.plots import plot_spectra, save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.quality import consensus_label_suspects, flag_shares, quality_flags

split = load_split()
X, y = split.X_train, split.y_train
labels = combined_label(y)
cells = X[AEZ_COL].astype(str) + " | " + X[MONTH_COL].astype(str)

**Skala prüfen:** Median der Reflektanz je Band im Blauen (400–500 nm) auf dem Trainingsteil.

In [ ]:
blue = next(region for region in SPECTRAL_REGIONS if region.name == "Blau")
blue_bands = [
    band
    for band, nm in zip(band_columns(X), wavelengths(X), strict=True)
    if blue.start_nm <= nm < blue.end_nm and X[band].notna().any()
]
X[blue_bands].median().round(1).rename("Median Reflektanz [%]").to_frame().T

## 1. Markierungen

In [ ]:
flags = quality_flags(X, y)
summary = pd.DataFrame({"Zeilen": flags.sum(), "Anteil": flags.mean()})
summary.round(3)

## 2. Wo liegen die Markierungen?

Pro Crop/Stage-Klasse und pro Aufnahme (AEZ | Monat), jeweils die zehn auffälligsten.

In [ ]:
flag_shares(flags, labels).head(10).round(3)

In [ ]:
flag_shares(flags, cells).head(10).round(3)

Wie unterscheiden sich die auffälligsten Aufnahmen vom Rest? Mittleres Spektrum ± Standardabweichung, getrennt nach „Aufnahme mit über 40 % markierten Zeilen“ und „übrige“.

In [ ]:
cell_shares = flags[ANY_FLAG_COL].groupby(cells).mean()
suspicious_cells = cell_shares[cell_shares > 0.4].index
spectra = X[band_columns(X)].assign(
    Aufnahme=cells.isin(suspicious_cells).map({True: "auffällig", False: "übrige"})
)
spectra_ax = plot_spectra(spectra, by="Aufnahme", title="Auffällige Aufnahmen vs. übrige")
save_doc_figure(spectra_ax.figure, "datenqualitaet_aufnahmen", MODEL_DOCS_FIGURES_DIR)
sorted(suspicious_cells)

**Befund Markierungen:**

- **Die Daten sind sauber:** keine extremen Dunst- oder Schatten-Zeilen und keine unplausiblen Kombinationen aus Monat × Kultur × Stadium.
- **5,5 % der Zeilen sind markiert,** fast alle als Klassen-Ausreißer (229) und einige als NDVI-Ausreißer (31).
- **Die Ausreißer bündeln sich in ganzen kleinen Aufnahmen:** AEZ 2 im Mai, Juni und Oktober sowie AEZ 6 im Mai. Dort sind 46–100 % der Zeilen markiert, zusammen 75 Zeilen.
- **Die Spektren dieser Aufnahmen** sind flach und hell: kaum Chlorophyll-Absorption, kaum Red Edge, hohes SWIR. So sieht trockener, wenig bewachsener Boden aus (Saisonrand bzw. trockene Region). Es sind also echte, abweichende Aufnahmen, keine Messfehler.

## 3. Häufen sich die Fehler in markierten Zeilen?

Out-of-fold-Vorhersagen der Referenz-SVM und von Extra Trees (beide kombinierte Klasse, zufällige Folds). Jede Zeile wird von einem Modell vorhergesagt, das sie nicht gesehen hat.

In [ ]:
base = PreprocessingConfig(use_spectral_standard_scale=True)
oof_svm = out_of_fold_predictions(
    make_combined_svm().set_params(estimator__C=300.0, estimator__gamma=0.003),
    RunConfig(name="quality_oof_svm", approach="combined", preprocessing=base, track=False),
)
oof_trees = out_of_fold_predictions(
    make_combined_extra_trees().set_params(
        estimator__max_features=0.3, estimator__min_samples_leaf=2
    ),
    RunConfig(name="quality_oof_trees", approach="combined", preprocessing=base, track=False),
)
error = combined_label(oof_svm.loc[y.index]).ne(labels)
error_rates = pd.DataFrame(
    {
        flag: {
            "Fehlerquote markiert": error[flags[flag]].mean(),
            "Fehlerquote nicht markiert": error[~flags[flag]].mean(),
            "markierte Zeilen": int(flags[flag].sum()),
        }
        for flag in flags.columns
    }
).T
error_rates.round(3)

## 4. Label-Verdacht (Konsens-Filter)

In [ ]:
flags[LABEL_SUSPECT_COL] = consensus_label_suspects([oof_svm, oof_trees], y)
suspects = flags[flags[LABEL_SUSPECT_COL]]
print(f"{len(suspects)} verdächtige Labels ({len(suspects) / len(flags):.1%})")
pd.DataFrame(
    {
        "Label": labels[suspects.index],
        "beide Modelle sagen": combined_label(oof_svm.loc[suspects.index]),
    }
).value_counts().head(15)

In [ ]:
suspect_share = flags[LABEL_SUSPECT_COL].groupby(labels).agg(["mean", "sum", "size"])
suspect_share.sort_values("mean", ascending=False).head(10).round(3)

## Fazit

| | Zeilen | Fehlerquote SVM (out-of-fold) |
| --- | --- | --- |
| markiert (Regeln) | 248 (5,5 %) | 25,4 % |
| nicht markiert | 4.223 | 10,1 % |
| Label-Verdacht (Konsens) | 281 (6,3 %) | 100 % (per Definition) |

- **Markierte Zeilen sind 2,5-mal so oft falsch, erklären aber nur gut 13 % der Fehler** (rund 63 von 490). Selbst ein perfektes Entfernen ließe den Großteil der Fehler übrig.
- **Die verdächtigen Labels sind überwiegend physikalisch mehrdeutige Fälle, keine falschen Labels:**
    - reifer Mais ↔ keimender Winterweizen (22), beide kaum grün,
    - Mais, Soja und Baumwolle in „Emerge“, wo der Boden das Spektrum dominiert,
    - benachbarte Stadien (Baumwolle Emerge ↔ Early_Mid, Soja Critical ↔ Late).
- **Folge für die Pipeline:** keine automatische Entfernung. Auffällige Aufnahmen kommen auch im Test vor (AEZ 2), und das Entfernen der verdächtigen Labels würde die schweren Fälle aus dem Training nehmen und die CV beschönigen. Ob das Entfernen der spektral markierten Zeilen *nur im Trainingsfold* hilft, prüft #197.
- **Für den Bericht:** Die verbleibenden Fehler sind keine Datenfehler, sondern Mehrdeutigkeit der Stadien mit viel Boden im Bild.